In [19]:
import requests
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
text = requests.get(url).text
words = text.splitlines()
print(words[:10])

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


In [20]:
b = {}
for w in words:
    ch = ['<S>'] + list(w) + ['<E']
    for ch1, ch2 in zip(ch, ch[1:]):
        biagram = (ch1, ch2)
        b[biagram] = b.get(biagram, 0) + 1

In [21]:
bi =  sorted(b.items(), key= lambda kv: -kv[1])
bi[:4]

[(('n', '<E'), 6763),
 (('a', '<E'), 6640),
 (('a', 'n'), 5438),
 (('<S>', 'a'), 4410)]

In [26]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos,'\n',stoi)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'} 
 {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}


In [ ]:
import torch
N = torch.zeros((27,27),dtype=torch.int32)

In [6]:
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1

In [7]:
p = N[0].float()
p = p / p.sum()
p

tensor([0.0000, 0.1377, 0.0408, 0.0481, 0.0528, 0.0478, 0.0130, 0.0209, 0.0273,
        0.0184, 0.0756, 0.0925, 0.0491, 0.0792, 0.0358, 0.0123, 0.0161, 0.0029,
        0.0512, 0.0642, 0.0408, 0.0024, 0.0117, 0.0096, 0.0042, 0.0167, 0.0290])

In [8]:
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item() #probability distribution
itos[ix]

'c'

In [9]:
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3,generator=g)
p = p / p.sum()
p

tensor([0.6064, 0.3033, 0.0903])

In [10]:
torch.multinomial(p,num_samples=100,replacement= True,generator= g)

tensor([1, 1, 2, 0, 0, 2, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 2, 0, 0,
        1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1,
        0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 1, 0,
        0, 1, 1, 1])

In [11]:
p.shape

torch.Size([3])

In [12]:
P = (N+1).float()
P /= P.sum(1, keepdim=True)

In [13]:
P[0]

tensor([3.1192e-05, 1.3759e-01, 4.0767e-02, 4.8129e-02, 5.2745e-02, 4.7785e-02,
        1.3038e-02, 2.0898e-02, 2.7293e-02, 1.8465e-02, 7.5577e-02, 9.2452e-02,
        4.9064e-02, 7.9195e-02, 3.5777e-02, 1.2321e-02, 1.6095e-02, 2.9008e-03,
        5.1154e-02, 6.4130e-02, 4.0830e-02, 2.4641e-03, 1.1759e-02, 9.6070e-03,
        4.2109e-03, 1.6719e-02, 2.9008e-02])

In [14]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement= True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print("".join(out))

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.


In [15]:
log_likelihood= 0
n= 0

for w in words:
  chs= ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1= stoi[ch1]
    ix2= stoi[ch2]
    prob= P[ix1,ix2]
    logprob= torch.log(prob)
    log_likelihood+= logprob
    n+= 1

print(f"{log_likelihood= }")
nll= -log_likelihood
print(f'{nll= }')
print(f'{nll/n = }')

log_likelihood= tensor(-559951.5625)
nll= tensor(559951.5625)
nll/n = tensor(2.4544)


In [27]:
# Goal is to maximize the liklihood of the data wrt model parameters
# which is equivalent to max(log-likelihood)
# and which is equivalent to min(neg_log-likelihood)
# which is equivalent to minimizing the mean(neg_log-likelihood)

In [16]:
# training set
xs, ys = [], []

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    xs.append(ix1)
    ys.append(ix2)

xs= torch.tensor(xs)
ys= torch.tensor(ys)

num = xs.nelement()

# Initializing the Model
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27,27), generator = g, requires_grad = True) # 27 neurons recieving 27 inputs

In [17]:
import torch.nn.functional as F

In [28]:
for k in range(500):
    #forward pass
    xenc = F.one_hot(xs, num_classes=27).float()  # input to the network
    logits = xenc @ W # predicting log_counts
    counts = logits.exp()
    probs = counts/ counts.sum(1,keepdims=True)
    # last 2 lines can be replaced by F.softmax(logits)
    loss = -probs[torch.arange(num),ys].log().mean() + 0.01*(W**2).mean()
    # loss + L2 regularization to prevent the W to be very large

    #backward pass
    W.grad = None # gradient setting to 0
    loss.backward()

    # Updating weights
    W.data += -0.80*W.grad

print(loss)

tensor(2.7396, grad_fn=<AddBackward0>)


In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(8):

  out = []
  ix = 0
  while True:

    # ----------
    # BEFORE:
    #p = P[ix]
    # ----------
    # NOW:
    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    # ----------

    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))


dexzmalegllurailezityha.
kllimittain.
llayn.
eria.
sade.
isan.
enkaviyni.
asu.
